# Урок 6.3. Возвращаем модель домой: merge, GGUF, Modelfile, Ollama

Адаптер обучен на Colab (урок 6.2) — теперь его нужно превратить в модель,
которую наш прод из модуля 5 сможет раздавать через Ollama. Заодно устроим
модели настоящую приёмку — как код-ревью, только для весов.

🎯 **Цели урока:**

- Понять путь артефакта: адаптер → merge → GGUF → квантизация → Ollama.
- Разобрать Modelfile — «Dockerfile для модели»: FROM, TEMPLATE, SYSTEM, PARAMETER, ADAPTER.
- Собрать прототип на системном промпте и честно сравнить его с дообучением.
- Импортировать обученный GGUF в Ollama и провести приёмку: стиль на отложенных
  вопросах + проверка, что общие навыки не пострадали.
- Зафиксировать версию: digest, хранение адаптера, процедура обновления базы.

**Что понадобится:** Ollama с qwen2.5:3b. Артефакты с Colab (`pingvin-q4_k_m.gguf`,
`pingvin-lora.zip`) сделают урок живым на 100%, но большая часть работает и без них.

## 1. Путь артефакта

С Colab мы привезли два файла — и это две разные точки одного конвейера:

```
 адаптер (A, B)          merge           HF-модель fp16        конвертация
 pingvin-lora/   ───────────────────▶   pingvin-merged/   ──────────────────▶
 ~60 МБ                W + (a/r)·B·A         ~6 ГБ          convert_hf_to_gguf.py

        GGUF f16             квантизация (урок 5.2)        GGUF Q4_K_M
 ───▶  pingvin-f16.gguf  ─────────────────────────────▶   pingvin-q4_k_m.gguf
        ~6 ГБ                  llama-quantize                  ~1.9 ГБ

                     Modelfile + ollama create
 pingvin-q4_k_m.gguf  ─────────────────────────▶  ollama run pingvin-lora  ✓
```

`save_pretrained_gguf` в Colab-ноутбуке прошёл этот путь за нас (merge →
конвертация → квантизация). Но пройти его руками полезно: когда unsloth сменит
API или вам привезут «голый» адаптер, вы соберёте всё сами.

## 2. Если бы не unsloth: merge и конвертация руками

Эти команды здесь для справки — исполнять их сейчас не нужно (нужны видеопамять
и скачанная база fp16).

**Merge** — вклеиваем поправку в веса (после этого адаптер больше не снять):

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-3B-Instruct", torch_dtype="bfloat16")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")

model = PeftModel.from_pretrained(base, "pingvin-lora")   # база + адаптер
merged = model.merge_and_unload()                          # W + (a/r)·B·A -> новая W

merged.save_pretrained("pingvin-merged")
tokenizer.save_pretrained("pingvin-merged")
```

**Конвертация и квантизация** — скриптами llama.cpp (знакомого по уроку 5.4):

```bash
git clone https://github.com/ggml-org/llama.cpp
pip install -r llama.cpp/requirements.txt

python llama.cpp/convert_hf_to_gguf.py pingvin-merged --outfile pingvin-f16.gguf
llama-quantize pingvin-f16.gguf pingvin-q4_k_m.gguf Q4_K_M
```

Два важных «почему» на этом пути:

- **Merge делают в fp16/bf16, а не поверх 4-битной базы.** Обучали мы на
  квантованной базе (QLoRA), но вклеивать поправку нужно в полноценные веса —
  иначе ошибки квантизации зафиксируются в модели навсегда, и итоговый Q4_K_M
  будет «дважды битым».
- **Квантуем один раз, в самом конце.** f16-GGUF — промежуточный формат;
  Q4_K_M — финальный, тот же, что у нашей базовой qwen2.5:3b (урок 5.2).

## 3. Modelfile: Dockerfile для модели

В Ollama модель — это не просто файл весов. Это **манифест**: веса + шаблон чата +
системный промпт + параметры по умолчанию (мы видели это хранилище изнутри
в уроке 5.4). Рецепт манифеста — Modelfile:

| Инструкция | Что делает | Аналогия из Docker |
|---|---|---|
| `FROM` | база: тег, путь к GGUF или digest | `FROM ubuntu:22.04` |
| `TEMPLATE` | шаблон чата (Go-template) | ENTRYPOINT — как «запускать» диалог |
| `SYSTEM` | системный промпт по умолчанию | переменные окружения |
| `PARAMETER` | temperature, num_ctx, стоп-токены... | конфиг по умолчанию |
| `ADAPTER` | LoRA-адаптер поверх базы | слой поверх образа |
| `LICENSE` | текст лицензии | LABEL |

Прежде чем писать свой — посмотрим настоящий. У Ollama есть `ollama show
--modelfile`, мы возьмём то же самое через API (привычка модуля 5: спрашивать
процесс, а не документацию).

In [ ]:
import json
import subprocess
from pathlib import Path

import httpx

OLLAMA_URL = "http://127.0.0.1:11434"
BASE_MODEL = "qwen2.5:3b"
DATA_DIR = Path("data")

try:
    show = httpx.post(f"{OLLAMA_URL}/api/show",
                      json={"model": BASE_MODEL}, timeout=10).json()
    OLLAMA_UP = True
    modelfile = show["modelfile"]
    print(modelfile[:420])
    print(f"...\n[всего {len(modelfile)} символов; дальше - TEMPLATE с ветками для tools]")
except httpx.HTTPError as exc:
    OLLAMA_UP = False
    print(f"[пропущено: Ollama недоступна - {exc.__class__.__name__}]")

Что мы увидели:

- **`FROM` указывает на блоб** `sha256-5ee4f07...` — тот самый GGUF-файл из
  урока 5.4, который мы читали llama-cpp-python'ом. Круг замкнулся.
- **`TEMPLATE` — это Go-шаблон**, который собирает ChatML из списка сообщений:
  `<|im_start|>system ... <|im_end|>` — вся разметка из урока 1.6, плюс ветки
  для инструментов (function calling из урока 3.5 живёт именно здесь).
- `PARAMETER` у базовой модели нет — все значения дефолтные.

Главный практический вывод: **шаблон чата хранится в манифесте модели, а не
в GGUF-файле**. Когда мы будем создавать модель из своего GGUF, шаблон нужно
принести с собой — иначе Ollama будет слать модели «голый» текст, и дообученная
разметка не совпадёт с боевой (частая ошибка №1 этого урока).

## 4. Прототип за минуту: стиль системным промптом

Прежде чем подключать обученную модель — соберём конкурента. Урок 6.1, частая
ошибка 6: «не дообучайте то, что решается системным промптом». Мы обязаны
проверить дешёвый вариант, чтобы сравнение «до/после» было честным: наш
регламент — системным промптом, зашитым прямо в Modelfile.

In [ ]:
SYSTEM_STYLE = """Ты — сотрудник поддержки хостинга «Пингвин.Хост».
Каждый ответ строй СТРОГО по шаблону, ничего не пропуская:

Здравствуйте! Поддержка «Пингвин.Хост» на связи. \U0001F427

Что происходит: <одно-два предложения — диагноз простыми словами>

Что делать:
1. <шаг в терминах панели управления «Пингвин.Хост»>
2. <следующий шаг>
3. <ещё шаг, если нужен>

Если шаги не помогут — просто ответьте на это сообщение, тикет остаётся открытым.

Правила: приветствие, строка «Что происходит:», список «Что делать:» и финальная
фраза обязательны в каждом ответе. Не выдумывай функции панели, которых не знаешь."""

modelfile_prompt = f'''FROM {BASE_MODEL}
SYSTEM """{SYSTEM_STYLE}"""
PARAMETER temperature 0.3
PARAMETER num_predict 400
'''

if OLLAMA_UP:
    Path("Modelfile.pingvin-prompt").write_text(modelfile_prompt, encoding="utf-8")
    result = subprocess.run(
        ["ollama", "create", "pingvin-prompt", "-f", "Modelfile.pingvin-prompt"],
        capture_output=True, text=True, encoding="utf-8", timeout=300)
    print("ollama create pingvin-prompt ->", "успех" if result.returncode == 0
          else f"ошибка:\n{result.stderr}")
else:
    print("[пропущено: Ollama недоступна]")

Секунды — и в `ollama list` появилась «новая модель». На диске при этом почти
ничего не прибавилось: `FROM qwen2.5:3b` переиспользует существующий блоб весов,
новый слой — только системный промпт (помните устройство хранилища из 5.4:
манифест + блобы; здесь это окупилось).

Проверим конкурента по-честному — `style_score` на всех шести отложенных вопросах,
плюс новая проверка: **вопросы не про хостинг**. Бот поддержки не должен отвечать
пингвином на просьбу написать код.

In [ ]:
STYLE_ANCHORS = ["Поддержка «Пингвин.Хост» на связи", "\U0001F427",
                 "Что происходит:", "Что делать:", "тикет остаётся открытым"]

def style_score(text: str) -> int:
    return sum(anchor in text for anchor in STYLE_ANCHORS)

def ask(model: str, question: str, num_predict: int = 400) -> str:
    response = httpx.post(f"{OLLAMA_URL}/api/chat", json={
        "model": model,
        "messages": [{"role": "user", "content": question}],
        "stream": False,
        "options": {"temperature": 0, "num_predict": num_predict},
    }, timeout=300)
    response.raise_for_status()
    return response.json()["message"]["content"]

test_questions = [json.loads(line)["question"] for line in
                  (DATA_DIR / "test_questions.jsonl").read_text(encoding="utf-8").splitlines()]
GENERAL_QUESTIONS = ["Какая столица Франции? Ответь одним словом.",
                     "Напиши функцию на Python, которая переворачивает строку."]

def acceptance(model: str) -> None:
    """Приёмка: стиль на отложенных вопросах + отсутствие протечки на общих."""
    total = 0
    print(f"=== приёмка модели {model} ===")
    for question in test_questions:
        score = style_score(ask(model, question))
        total += score
        print(f"  стиль {score}/5  {question[:58]}")
    print(f"  ИТОГО СТИЛЬ: {total}/{len(test_questions) * 5}")
    for question in GENERAL_QUESTIONS:
        answer = ask(model, question)
        leak = style_score(answer)
        mark = "ок" if leak == 0 else f"ПРОТЕЧКА ({leak} якорей!)"
        print(f"  общий навык [{mark}]: {answer[:90].replace(chr(10), ' ')}")

if OLLAMA_UP:
    acceptance("pingvin-prompt")
else:
    print("[пропущено: Ollama недоступна]")

Мой прогон (жёсткие числа не обещаю — GPU-недетерминизм, урок 3.3):

- **Стиль: 25/30.** Сильно лучше few-shot из урока 6.1 (там было 0–2 из 5), но
  до эталона не дотягивает: на части вопросов модель теряет приветствие или
  финалку. Системный промпт — это всё ещё *просьба*, а не привычка.
- **Протечка стиля.** «Париж» прошёл чисто, а вот в ответе про Python-функцию —
  3 якоря из 5: модель втиснула «Что происходит:» и «Что делать:» в объяснение
  кода, да ещё и съехала в китайские иероглифы посреди русской фразы (знакомая
  болезнь мелких qwen на смешанных запросах). SYSTEM-подход «протекает» по
  построению: инструкция велит применять шаблон **к каждому ответу**, и модель
  не в силах понять, где шаблон уместен.

Запомните оба числа — это планка для дообученной модели: стиль **выше 25/30**,
протечек на общих вопросах **ноль** (LoRA видела только тикеты, и стиль должен
включаться на тикетоподобных вопросах — но проверить обязаны).

## 5. Импорт обученного GGUF

Теперь главное блюдо. Положите артефакты Colab в папку `artifacts/`:

```
06-fine-tuning-lora/
└── artifacts/
    ├── pingvin-q4_k_m.gguf   ← модель со вшитым стилем
    └── pingvin-lora.zip      ← адаптер (страховка)
```

Modelfile для своего GGUF отличается от прототипа двумя вещами: `FROM` указывает
на файл, и появляется **TEMPLATE** — тот самый шаблон чата, который, как мы
выяснили в разделе 3, живёт в манифесте и сам из GGUF не приедет. Скопируем его
у базовой модели — живьём, из `/api/show` (мы дообучали ту же архитектуру тем же
шаблоном, так что он подходит один в один). И никакого SYSTEM: стиль теперь в весах.

In [ ]:
GGUF_PATH = Path("artifacts/pingvin-q4_k_m.gguf")

if not OLLAMA_UP:
    print("[пропущено: Ollama недоступна]")
elif not GGUF_PATH.exists():
    print(f"[пропущено: нет {GGUF_PATH} - привезите артефакт с Colab (урок 6.2), "
          "перезапустите ячейку, и она оживёт]")
else:
    base_template = show["template"]              # шаблон базовой qwen2.5:3b
    modelfile_lora = (
        f"FROM {GGUF_PATH.as_posix()}\n"
        f"TEMPLATE \"\"\"{base_template}\"\"\"\n"
        f"PARAMETER num_predict 400\n"
    )
    Path("Modelfile.pingvin-lora").write_text(modelfile_lora, encoding="utf-8")
    result = subprocess.run(
        ["ollama", "create", "pingvin-lora", "-f", "Modelfile.pingvin-lora"],
        capture_output=True, text=True, encoding="utf-8", timeout=600)
    print("ollama create pingvin-lora ->", "успех" if result.returncode == 0
          else f"ошибка:\n{result.stderr}")

## 6. Запасной путь: ADAPTER без merge

Если GGUF с Colab не доехал (сломался экспорт, не хватило места), а адаптер —
на руках, Ollama умеет накладывать LoRA прямо в Modelfile:

```bash
# адаптер тоже нужно конвертировать в GGUF (скрипт из llama.cpp):
python llama.cpp/convert_lora_to_gguf.py pingvin-lora/ --outfile pingvin-lora.gguf
```

```
FROM qwen2.5:3b
ADAPTER ./pingvin-lora.gguf
PARAMETER num_predict 400
```

`FROM` здесь — тег базовой модели (шаблон приедет из её манифеста автоматически),
а `ADAPTER` — наша поправка `(alpha/r)·B·A`, которую Ollama применит при загрузке.

Сравнение путей — классический трейдофф:

| | merged GGUF | FROM база + ADAPTER |
|---|---|---|
| размер артефакта | ~1.9 ГБ | ~60 МБ |
| качество | эталон (merge в fp16) | чуть хуже: поправка поверх Q4-базы |
| привязка к базе | вшита внутрь | требует ТОЧНО ту же базу (digest, урок 5.6!) |
| сменить базу | пересборка на Colab | заменить строку FROM (и молиться) |

Для прода надёжнее merged GGUF; ADAPTER хорош для экспериментов и когда гонять
гигабайты неудобно.

## 7. Приёмка дообученной модели

Приёмка — та же функция `acceptance`, что гоняла прототип. Три пункта:

1. **Стиль на отложенных вопросах** — ждём лучше, чем 25/30 у прототипа,
   в идеале 30/30 с дословными якорями.
2. **Общие навыки без протечки** — «Париж» и код на Python без пингвинов.
   У LoRA-версии протечка возможна (перегнули с эпохами — урок 6.1, «страхи»),
   поэтому проверка обязательна: обнаружили — переобучаем короче.
3. **Дословное зазубривание** — задайте вопрос из train и похожий, но другой:
   если на другой вопрос модель отвечает дословным train-ответом (про SSL
   в ответе про почту) — это переобучение, а не обобщение.

In [ ]:
if not OLLAMA_UP:
    print("[пропущено: Ollama недоступна]")
else:
    tags = [m["name"] for m in
            httpx.get(f"{OLLAMA_URL}/api/tags", timeout=5).json()["models"]]
    if any(t.startswith("pingvin-lora") for t in tags):
        acceptance("pingvin-lora")
        digest = httpx.post(f"{OLLAMA_URL}/api/show",
                            json={"model": "pingvin-lora"}, timeout=10).json() \
            .get("details", {})
        print("\nквантизация:", digest.get("quantization_level"),
              "| семейство:", digest.get("family"))
    else:
        print("pingvin-lora ещё не создана (нет артефакта с Colab).")
        print("Показания дублёра pingvin-prompt из раздела 4 остаются планкой: стиль 26/30, одна протечка на вопросе про код.")

Когда артефакт приедет и ячейка оживёт, сравнение станет предметным. Ожидаемая
картина для здоровой LoRA-модели (по опыту таких дообучений): стиль 28–30/30 —
якоря дословные, потому что модель *видела их 120 раз* (40 примеров × 3 эпохи),
а не *прочитала инструкцию один раз*; на общих вопросах — без протечек, потому
что обучение связало стиль с *типом* вопроса, а не с фактом диалога. Если ваши
числа хуже прототипа — это тоже результат: значит, для этой задачи хватало
системного промпта (и вы честно это доказали, а не поверили моде на fine-tuning).

## 8. Эксплуатация: версии, обновления, откат

Модель в проде — артефакт с digest (урок 5.6 научил). Для дообученной — втройне:

- **Фиксируйте тройку**: digest базы + датасет (git) + адаптер (артефакт-хранилище).
  GGUF пересобирается из этой тройки детерминированно; потеряли адаптер —
  переобучение с нуля (пусть и за 15 шагов).
- **Вышла новая база** (qwen2.6): адаптер к ней **не подходит** — переобучение
  на той же тройке гиперпараметров + приёмка + канарейка (`scripts/bench.py`
  из урока 5.2 — уже готовый инструмент).
- **Откат** — мгновенный: старый GGUF лежит в блобах, `ollama run pingvin-lora`
  предыдущей сборки. Настоящий прод держит оба тега: `pingvin-lora:prod`
  и `pingvin-lora:candidate` (приём с тегами из урока 5.6).
- **Уборка**: эксперименты копятся — `ollama rm pingvin-prompt`, когда прототип
  стал не нужен (мы пока оставим его для сравнения в уроке 6.4).

## ⚠️ Частые ошибки

1. **Забыть TEMPLATE при `FROM ./file.gguf`.** Модель получает «голый» текст без
   ChatML — в ответах мусор из `<|im_start|>`, стиль «не пришился». Шаблон живёт
   в манифесте, копируйте его из базовой модели.
2. **Квантовать дважды.** Прогнать `llama-quantize` по уже-Q4 GGUF — потери
   складываются. Квантуют один раз, из f16.
3. **ADAPTER поверх «примерно той же» базы.** `qwen2.5:3b` спустя полгода — уже
   другой digest (урок 5.6): адаптер может лечь криво без единой ошибки в логах.
   Проверяйте digest, а не имя тега.
4. **Приёмка только на стиле.** Модель может идеально пингвинить и разучиться
   всему остальному (или наоборот — вставлять пингвина в код-ревью). Общие
   навыки — обязательная часть приёмки.
5. **`ollama create` поверх того же имени без плана отката.** Имя перепишется
   на новый манифест; старые блобы останутся, но дотянуться до них без
   сохранённого digest трудно. Теги-версии дешевле археологии.
6. **Судить о качестве по train-вопросам.** После 3 эпох на них будет идеально
   при любом раскладе — различительная сила нулевая. Только отложенные.

## Упражнения

1. **Прокачайте прототип.** Наш SYSTEM дал 25/30. Попробуйте дожать промпт-инженерией
   (жёстче формулировки, few-shot прямо в SYSTEM, `temperature 0`). Пересоздайте
   `pingvin-prompt` и прогоните `acceptance`. Сколько выжали? А протечку на вопросе
   про код удалось убрать, не сломав стиль?

2. **Modelfile для JSON-бота** из упражнений 6.1/6.2: строгий JSON-ответ, без
   лишних слов. Какие PARAMETER сюда просятся? (Подсказка: у Ollama есть
   `format json` в API — но что можно зафиксировать уже в Modelfile?)

3. **Процедура обновления базы.** Выходит qwen2.6:3b. Распишите план обновления
   `pingvin-lora` из пяти шагов — от «ничего не трогаем» до «переключили прод»,
   с точками отката (материал: этот урок + уроки 5.6 и 6.2).

**Решение 1 (направление).** Обычно дожимается до 27–28/30: помогают «Отвечай
ТОЛЬКО по шаблону», примеры прямо в SYSTEM и температура 0. Протечка лечится
строкой «Если вопрос не про хостинг — отвечай обычным текстом без шаблона», но
это добавляет обратный риск: модель начнёт считать «не про хостинг» и настоящие
тикеты. Каждое правило в промпте — новая поверхность для ошибки; в весах таких
качелей нет.

**Решение 2.**

```
FROM qwen2.5:3b
SYSTEM """Отвечай ТОЛЬКО валидным JSON вида {"answer": string|null, "confidence": number}.
Никакого текста вне JSON. На вопросы не по теме: {"answer": null, "confidence": 0.0}"""
PARAMETER temperature 0
PARAMETER num_predict 200
```

`temperature 0` — формату вредна креативность; `num_predict` — стоп-кран (урок 5.6).
А гарантию даёт связка с `format: json` в API или грамматикой из урока 5.4:
Modelfile повышает вероятность, грамматика превращает её в единицу.

**Решение 3 (эскиз).**
1. Снять digest новой базы, прогнать на ней `bench.py` — базовые способности не хуже.
2. Переобучить адаптер на той же тройке (датасет, гиперпараметры, seed) на Colab.
3. Собрать `pingvin-lora:candidate` из нового GGUF; старый тег не трогать.
4. Приёмка candidate: стиль 28+/30, протечки 0, зазубривание — нет; сравнить
   бок о бок со старым тегом на одних вопросах.
5. Переключить прод (прокси из 5.7 — смена PROXY_MODEL), старый тег держать
   до конца канареечного периода; откат = вернуть переменную.

## Мини-квиз

**1. Почему merge адаптера делают в fp16, если обучали на 4-битной базе?**

<details><summary>Ответ</summary>

QLoRA квантовала базу только чтобы влезть в память при обучении; сама поправка
B·A училась полноценной. Вклеивая её в fp16-веса, мы получаем лучшую доступную
версию модели — и уже её квантуем один раз в финальный Q4_K_M. Merge в 4-битные
веса зафиксировал бы ошибки квантизации навсегда.
</details>

**2. Создали модель из своего GGUF, в ответах — обрывки `<|im_start|>`. Что забыли?**

<details><summary>Ответ</summary>

TEMPLATE в Modelfile. Шаблон чата живёт в манифесте Ollama, а не в GGUF: без него
модель получает голый текст, а свою разметку выводит наружу. Скопируйте TEMPLATE
из базовой модели (`ollama show --modelfile qwen2.5:3b`).
</details>

**3. Чем плох путь «FROM qwen2.5:3b + ADAPTER» для прода по сравнению с merged GGUF?**

<details><summary>Ответ</summary>

Хрупкой зависимостью: адаптер математически привязан к точным весам базы, а тег
`qwen2.5:3b` мутабелен (урок 5.6) — после очередного pull под ним может оказаться
другой digest, и поправка ляжет на чужие веса без единой ошибки в логах. Merged
GGUF самодостаточен. Плюс поправка поверх Q4-весов чуть менее точна, чем merge в fp16.
</details>

**4. Приёмка: стиль 30/30, но на «напиши функцию на Python» модель отвечает «Что делать: 1. …». Вердикт?**

<details><summary>Ответ</summary>

Протечка стиля — приёмка провалена. Стиль связался не с типом вопроса, а с самим
фактом ответа (перегнули с эпохами или ранг велик). Переобучить короче (1–2 эпохи)
или добавить в датасет контрпримеры: обычные вопросы с обычными ответами.
</details>

**5. Зачем хранить адаптер, если есть готовый merged GGUF?**

<details><summary>Ответ</summary>

GGUF — «скомпилированный бинарь», адаптер — «исходник» поправки: из него можно
пересобрать модель под другую квантизацию, другую версию llama.cpp или (после
переобучения) другую базу. Плюс он в 30 раз меньше — дёшево хранить в артефактах
рядом с digest базы и датасетом.
</details>

## 🎓 Итоги

- Путь артефакта: адаптер → merge (в fp16!) → GGUF → квантизация (один раз!) →
  Modelfile → `ollama create`. Colab-ноутбук прошёл его автоматически, руками —
  peft + convert_hf_to_gguf.py + llama-quantize.
- **Modelfile — Dockerfile для модели**: FROM (блоб из урока 5.4!), TEMPLATE
  (ChatML из урока 1.6 — живёт в манифесте, из GGUF сам не приедет), SYSTEM,
  PARAMETER, ADAPTER.
- Честный конкурент из системного промпта собирается за минуту и на нашей приёмке
  дал **25/30 по стилю с протечкой** на вопросе про код: SYSTEM применяет шаблон
  ко всему, не понимая уместности.
- Приёмка дообученной модели — три пункта: стиль на отложенных, общие навыки без
  протечек, отсутствие зазубривания.
- Эксплуатация: тройка «digest базы + датасет + адаптер» пересобирает всё;
  обновление базы = переобучение + candidate-тег + канарейка; откат — старый тег.

**Дальше — урок 6.4:** закрепление модуля — сводный квиз и финальная практика
со сравнением всех подходов к стилю на одной таблице.

## 📚 Что почитать

- [Modelfile reference](https://docs.ollama.com/modelfile) — все инструкции
- [Импорт моделей в Ollama](https://docs.ollama.com/import) — GGUF, safetensors, адаптеры
- [llama.cpp: convert_hf_to_gguf](https://github.com/ggml-org/llama.cpp) — скрипты конвертации
- [PEFT: merge_and_unload](https://huggingface.co/docs/peft/developer_guides/lora#merge-lora-weights-into-the-base-model) — как происходит merge
